# 21. ObservableGate, Decoder-Input Correctness

## 1. The pain

Every other notebook here prices a circuit *before* you run it. This one checks something subtler: whether the **decoder input** for an error-correction experiment is even well-formed before you spend a run decoding it.

A stim Detector Error Model (DEM) error mechanism carries three things: the **detectors** it flips (`H`), the **logical observables** it flips (`L`), and a **probability** (`p`). A decoder predicts the logical *frame* `L` from the detector *symptom* `H`. A common preprocessing step collapses a DEM into a check matrix by grouping mechanisms by their detector signature. If two mechanisms are **detector-identical but logical-distinct**, that merge silently picks one logical mask and throws the other away, and the logical frame is gone before the decoder ever sees it.

`ObservableGate` is a correctness preflight for exactly that hazard. It is not a performance optimization and it does not claim any decoder is broken; it detects an unsafe decoder-input construction and offers an observable-preserving fix.

In [1]:
import os
import tempfile

# This whole notebook works on stim DEMs, which need the [ising] extra.
# Guard the import so the notebook degrades gracefully instead of crashing.
try:
    import stim

    HAVE_STIM = True
except ImportError:
    HAVE_STIM = False

import numpy as np

import qb_compiler
from qb_compiler.observable_gate import (
    audit_dem,
    audit_matrices,
    canonicalize_dem,
    preflight_dem_gate,
    ObservableMaskCollapseError,
    ObservableAuditResult,
)

print("qb-compiler", qb_compiler.__version__)
print("stim available:", HAVE_STIM, "(install: pip install 'qb-compiler[ising]')")

qb-compiler 0.7.0.post1.dev0+gba6705ac8.d20260612
stim available: True (install: pip install 'qb-compiler[ising]')


## 2. The hazard, in two lines

The smallest possible witness is two error mechanisms that flip the same detector but disagree on the logical observable:

```
error(0.01) D0
error(0.01) D0 L0
```

Both flip detector `D0`; only the second flips logical `L0`. A detector-signature-only canonicalization would merge these into one column and lose the `L0` information. `audit_dem` flags it.

In [2]:
if HAVE_STIM:
    witness = stim.DetectorErrorModel('''
        error(0.01) D0
        error(0.01) D0 L0
    ''')
    result = audit_dem(witness)
    print(result)
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

ObservableGate DEM audit
  raw mechanisms             : 2
  unique detector signatures : 1
  unique detector+obs masks  : 2
  mixed detector groups      : 1
  mixed probability mass     : 0.020000
  worst mask ratio           : 1.0000
  max collapse bucket        : 2
  decomposed DEM             : False
  status: FAIL
  recommendation: detector-identical mechanisms carry conflicting masks; canonicalize by (detectors, observables) or preserve P(L|H), never merge by detector alone.


`status: FAIL`. Read the diagnostics: there is **1 unique detector signature** but **2 unique detector+observable masks**, so one detector pattern carries two different logical frames. The `worst mask ratio` of `1.0000` means the two competing masks are equally probable, maximally ambiguous, and the `mixed probability mass` (`0.02`) is the total weight sitting in that ambiguous group. The `recommendation` spells out the fix: never merge by detector alone.

## 3. The invariant, with no stim

The core check is pure linear algebra and needs no stim, handy for unit tests or environments without the extra. Group the mechanism columns by detector signature `H`; the construction is unsafe when

```
unique(H)  <  unique(H, L)
```

`audit_matrices` takes a check matrix `(n_detectors, n)`, an observable-mask matrix `(n_observables, n)`, and per-mechanism priors. Here both columns flip detector `D0` (`H` row all ones), but only the second flips `L0`.

In [3]:
check_matrix = np.array([[1, 1]], dtype=np.uint8)   # 1 detector, both columns flip it
obs_matrix = np.array([[0, 1]], dtype=np.uint8)     # 1 observable, only column 2 flips it
priors = np.array([0.01, 0.01])

r = audit_matrices(check_matrix, obs_matrix, priors)
print("status                     :", r.status)
print("unique detector signatures :", r.unique_detector_sigs)
print("unique detector+obs masks  :", r.unique_detector_obs_sigs)
print("unsafe_to_merge_by_detector:", r.unsafe_to_merge_by_detector)

status                     : FAIL
unique detector signatures : 1
unique detector+obs masks  : 2
unsafe_to_merge_by_detector: True


Same verdict as the DEM, with nothing but numpy. This is the invariant the whole gate is built on.

## 4. Real codes audit PASS

The hazard is real but *narrow*. The standard production paths are clean: a rotated surface-code memory has no detector-identical / logical-distinct mechanisms, so it audits `PASS` at every distance.

In [4]:
if HAVE_STIM:
    print(f"{'code':<28s} {'mechanisms':>10s} {'uniq(H)':>8s} {'uniq(H,L)':>10s} {'status':>7s}")
    print("-" * 67)
    for d in (3, 5, 7):
        circ = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            distance=d,
            rounds=d,
            after_clifford_depolarization=0.005,
            before_measure_flip_probability=0.005,
            after_reset_flip_probability=0.005,
        )
        dem = circ.detector_error_model(decompose_errors=False)
        a = audit_dem(dem)
        print(
            f"surface_code d={d:<2d} (rmz)        {a.n_mechanisms:>10d} "
            f"{a.unique_detector_sigs:>8d} {a.unique_detector_obs_sigs:>10d} {a.status:>7s}"
        )
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

code                         mechanisms  uniq(H)  uniq(H,L)  status
-------------------------------------------------------------------
surface_code d=3  (rmz)               219      219        219    PASS
surface_code d=5  (rmz)              1677     1677       1677    PASS


surface_code d=7  (rmz)              6023     5471       5471    PASS


`unique(H) == unique(H, L)` on every row, so no detector signature carries two masks: `PASS`. Surface and repetition codes and the full bivariate-bicycle / Gross family ([[72,12,6]] through [[288,12,18]], X and Z basis) all pass. The measured harm, a ~60% relative logical-error-rate inflation, shows up on graphlike DEMs that genuinely have detector-identical / logical-distinct mechanisms (e.g. `color_code:memory_xyz` d3). ObservableGate's job is to tell the two cases apart.

## 5. The fix: observable-preserving canonicalization

If you must canonicalize, do it by `(H, L)`: merge only **exact** `(detectors, observables)` duplicates (XOR-combining their probabilities) and keep detector-identical-but-logical-distinct pairs as separate columns. Nothing is erased. Build a DEM with an exact duplicate (should merge) plus a logical-distinct twin (must be preserved):

In [5]:
if HAVE_STIM:
    dem = stim.DetectorErrorModel('''
        error(0.01) D0
        error(0.02) D0
        error(0.01) D0 L0
    ''')
    safe = canonicalize_dem(dem)
    print(f"raw mechanisms  : {dem.num_errors}")
    print(f"canon mechanisms: {safe.num_errors}")
    print("canonical DEM:")
    print(safe)
    print()
    print("audit after canonicalize:", audit_dem(safe).status,
          "(the (H, L) ambiguity is intrinsic and preserved, never erased)")
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

raw mechanisms  : 3
canon mechanisms: 2
canonical DEM:
error(0.02959999999999999784) D0
error(0.01000000000000000021) D0 L0

audit after canonicalize: FAIL (the (H, L) ambiguity is intrinsic and preserved, never erased)


Three mechanisms became two: the two `D0`-only errors (`0.01` and `0.02`) XOR-combined into a single `D0` mechanism, while the `D0 L0` mechanism stayed separate. The distinct `(detector, mask)` pair survives. The audit still reports `FAIL`, and that is correct: the ambiguity is a real property of this DEM, so the canonical form *preserves* it rather than pretending it is gone. The actionable signal is *do not feed this to a detector-signature-only merge path*, not *this is now safe to merge*.

## 6. The receipt

`audit_dem` returns an `ObservableAuditResult`, a frozen dataclass you can log, assert on, or attach to a compilation receipt. `qec_preflight()` attaches one automatically as its `observable_audit` field.

In [6]:
if HAVE_STIM:
    witness = stim.DetectorErrorModel("error(0.01) D0\nerror(0.01) D0 L0")
    res = audit_dem(witness)
    print("=== ObservableAuditResult fields ===")
    print(f"n_mechanisms              : {res.n_mechanisms}")
    print(f"unique_detector_sigs      : {res.unique_detector_sigs}")
    print(f"unique_detector_obs_sigs  : {res.unique_detector_obs_sigs}")
    print(f"mixed_groups              : {res.mixed_groups}")
    print(f"mixed_mass                : {res.mixed_mass:.6f}")
    print(f"worst_mask_ratio          : {res.worst_mask_ratio:.4f}")
    print(f"max_group                 : {res.max_group}")
    print(f"decomposed                : {res.decomposed}")
    print(f"status                    : {res.status}")
    print(f"ok (property)             : {res.ok}")
    print(f"unsafe_to_merge_by_detector: {res.unsafe_to_merge_by_detector}")
    print(f"recommendation()          : {res.recommendation()}")
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

=== ObservableAuditResult fields ===
n_mechanisms              : 2
unique_detector_sigs      : 1
unique_detector_obs_sigs  : 2
mixed_groups              : 1
mixed_mass                : 0.020000
worst_mask_ratio          : 1.0000
max_group                 : 2
decomposed                : False
status                    : FAIL
ok (property)             : False
unsafe_to_merge_by_detector: True
recommendation()          : detector-identical mechanisms carry conflicting masks; canonicalize by (detectors, observables) or preserve P(L|H), never merge by detector alone.


## 7. The gate

`preflight_dem_gate` turns the audit into a hard stop: it raises `ObservableMaskCollapseError` on `FAIL` (and on `WARN` with `strict=True`), otherwise it returns the result. Drop it into a preflight or CI step to block decoding of a DEM whose detector-only canonicalization would erase the logical frame.

In [7]:
if HAVE_STIM:
    # PASS path: a clean surface-code DEM returns its audit result.
    clean = stim.Circuit.generated(
        "surface_code:rotated_memory_z", distance=3, rounds=3,
        after_clifford_depolarization=0.005,
        before_measure_flip_probability=0.005,
        after_reset_flip_probability=0.005,
    ).detector_error_model(decompose_errors=False)
    ok = preflight_dem_gate(clean)
    print("clean surface-code DEM -> gate returned:", ok.status)

    # FAIL path: the witness raises and blocks the run.
    witness = stim.DetectorErrorModel("error(0.01) D0\nerror(0.01) D0 L0")
    try:
        preflight_dem_gate(witness)
    except ObservableMaskCollapseError as exc:
        print("witness DEM            -> blocked:", exc)
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

clean surface-code DEM -> gate returned: PASS
witness DEM            -> blocked: observable-mask collapse risk (FAIL): 1 detector-identical / logical-distinct group(s), 0.0200 probability mass. Canonicalize by (detectors, observables) or use an observable-preserving path.


## 8. From the command line

The same audit is a CLI command, so it can gate a pipeline with no Python at all. `qbc dem-audit` exits `0` for PASS, `2` for FAIL (and `1` for WARN with `--strict`), CI-safe. `qbc dem-canonicalize` writes the observable-preserving form. We write two DEMs to a temp dir and run the CLI as a subprocess so the exit codes are visible.

In [8]:
import shutil
import subprocess

qbc = shutil.which("qbc")
if HAVE_STIM and qbc:
    tmp = tempfile.mkdtemp(prefix="obsgate_")
    witness_path = os.path.join(tmp, "witness.dem")
    surface_path = os.path.join(tmp, "surface.dem")
    safe_path = os.path.join(tmp, "witness_safe.dem")

    stim.DetectorErrorModel("error(0.01) D0\nerror(0.01) D0 L0").to_file(witness_path)
    stim.Circuit.generated(
        "surface_code:rotated_memory_z", distance=3, rounds=3,
        after_clifford_depolarization=0.005,
        before_measure_flip_probability=0.005,
        after_reset_flip_probability=0.005,
    ).detector_error_model(decompose_errors=False).to_file(surface_path)

    def run(*args):
        proc = subprocess.run([qbc, *args], capture_output=True, text=True)
        return proc.returncode, proc.stdout.strip()

    rc, out = run("dem-audit", surface_path)
    print(f"$ qbc dem-audit surface.dem      -> exit {rc} (PASS)")
    print(out.splitlines()[-1])
    print()

    rc, out = run("dem-audit", witness_path)
    print(f"$ qbc dem-audit witness.dem      -> exit {rc} (FAIL, blocks CI)")
    print(out.splitlines()[-1])
    print()

    rc, out = run("dem-canonicalize", witness_path, "-o", safe_path)
    print(f"$ qbc dem-canonicalize witness.dem -o safe.dem -> exit {rc}")
    print(out)
elif not qbc:
    print("skipped: 'qbc' console script not on PATH (pip install qb-compiler)")
else:
    print("skipped: requires the [ising] extra (pip install 'qb-compiler[ising]')")

$ qbc dem-audit surface.dem      -> exit 0 (PASS)
  recommendation: detector-only canonicalization appears observable-safe.



$ qbc dem-audit witness.dem      -> exit 2 (FAIL, blocks CI)
  recommendation: detector-identical mechanisms carry conflicting masks; canonicalize by (detectors, observables) or preserve P(L|H), never merge by detector alone.



$ qbc dem-canonicalize witness.dem -o safe.dem -> exit 0
observable-preserving DEM written to /tmp/obsgate_8h50d1o3/witness_safe.dem
  mechanisms: 2 -> 2
  status:     FAIL -> FAIL (distinct (detector, mask) pairs preserved)


## 9. Close

`ObservableGate` is a small, honest correctness check: it does not claim a decoder is wrong, and it does not slow anything down. It answers one question, *is this DEM safe to canonicalize by detector signature, or would that erase the logical frame?*, three ways: the Python `audit_dem` / `audit_matrices` invariant, the `preflight_dem_gate` hard stop, and the `qbc dem-audit` CI exit code. Standard surface / repetition / Gross-family DEMs pass; the graphlike detector-identical / logical-distinct case fails loudly, before a run is spent on it.